In [ ]:
%load_ext memprofiler
import dask.array as da
import numba
import numpy as np
import zarr
from numcodecs import Blosc

import blosc2

In [ ]:
N = 20_000

# For best speed
# blosc2.cparams_dflts["codec"] = blosc2.Codec.BLOSCLZ
blosc2.cparams_dflts["codec"] = blosc2.Codec.LZ4
blosc2.cparams_dflts["clevel"] = 1
# compressor = Blosc(cname='blosclz', clevel=5, shuffle=Blosc.SHUFFLE)
compressor = Blosc(cname="lz4", clevel=1, shuffle=Blosc.SHUFFLE)

In [ ]:
%%time
na = np.linspace(0, 1, N * N).reshape(N, N)
a = blosc2.asarray(na)
za = zarr.array(na, compressor=compressor, zarr_format=2, chunks=a.chunks)
nb = np.linspace(1, 2, N * N).reshape(N, N)
b = blosc2.asarray(nb)
zb = zarr.array(nb, compressor=compressor, zarr_format=2, chunks=b.chunks)
nc = np.linspace(-10, 10, N * N).reshape(N, N)
c = blosc2.asarray(nc)
zc = zarr.array(nc, compressor=compressor, zarr_format=2, chunks=c.chunks)

In [ ]:
%%time
# Expression (blosc2 form)
# expr = (a * 2 + b > c)
# expr = ((a ** 3 + blosc2.sin(c * 2)) < b)
expr = ((a**3 + blosc2.sin(c * 2)) < b) & (c > 0)
# numexpr form
# sexpr = "(na * 2 + nb > nc)"
# sexpr = "((na ** 3 + sin(nc * 2)) < nb)"
sexpr = "((na ** 3 + sin(nc * 2)) < nb) & (nc > 0)"

In [ ]:
%%mprof_run 1.LazyArray::compute-LZ4-1
# Evaluate and get a NDArray as result
out = expr.compute()

In [ ]:
out.info

In [ ]:
@numba.jit(parallel=True)
def func_expr(inputs_tuple, output, offset):
    a = inputs_tuple[0]
    b = inputs_tuple[1]
    c = inputs_tuple[2]
    for i in numba.prange(a.shape[0]):
        for j in numba.prange(a.shape[1]):
            # expr = (a[i, j] * 2 + b[i, j] > c[i, j])
            # expr = ((a[i, j] ** 3 + np.sin(c[i, j] * 2)) < b[i, j])
            expr = ((a[i, j] ** 3 + np.sin(c[i, j] * 2)) < b[i, j]) and (c[i, j] > 0)
            output[i, j] = expr
    output[:] = expr


lzyudf = blosc2.lazyudf(func_expr, (a, b, c), np.bool_)

In [ ]:
%%mprof_run 1.LazyArray::getitem-LZ4-1
# Evaluate and get a NDArray as result
out_ = expr[:]

In [ ]:
%%time
# Expression (dask form)
da_ = da.from_zarr(za)
db = da.from_zarr(zb)
dc = da.from_zarr(zc)
# dexpr = (da_ * 2 + db > dc)
# dexpr = ((da_ ** 3 + da.sin(dc * 2)) < db)
dexpr = ((da_**3 + da.sin(dc * 2)) < db) & (dc > 0)
scheduler = "single-threaded" if blosc2.nthreads == 1 else "threads"

In [ ]:
%%mprof_run 2.Dask::to_zarr-LZ4-1
zres = zarr.open(shape=(N, N), dtype=dexpr.dtype, compressor=compressor, zarr_format=2, chunks=a.chunks)
#with dask.config.set(scheduler=scheduler):
with dask.config.set(scheduler=scheduler, num_workers=blosc2.nthreads):
    da.to_zarr(dexpr, zres)

In [ ]:
%%mprof_run 2.Dask::compute-LZ4-1
#with dask.config.set(scheduler=scheduler):
with dask.config.set(scheduler=scheduler, num_workers=blosc2.nthreads):
    nres = dexpr.compute()

In [ ]:
%%time
# Expression (dask form, no compr)
da_ = da.from_array(na)
db = da.from_array(nb)
dc = da.from_array(nc)
# dexpr = (da_ * 2 + db > dc)
# dexpr = ((da_ ** 3 + da.sin(dc * 2)) < db)
dexpr = ((da_**3 + da.sin(dc * 2)) < db) & (dc > 0)
scheduler = "single-threaded" if blosc2.nthreads == 1 else "threads"

In [ ]:
%%mprof_run 3.NumExpr
# Evaluate with numexpr
out1 = ne.evaluate(sexpr)

In [ ]:
%%mprof_run 4.Numba
out2 = np.empty(out.shape, dtype=out.dtype)
func_expr((na, nb, nc), out2, 0)

In [ ]:
%%mprof_run 5.NumPy
# Evaluate with numpy
#out = (na * 2 + nb > nc) & (nc > 0)
#out = ((na ** 3 + np.sin(nc * 2)) < nb)
out = ((na ** 3 + np.sin(nc * 2)) < nb) & (nc > 0)

In [ ]:
%mprof_plot .* -t "AMD 9800X3D -- Number of threads: {blosc2.nthreads}"